# A2 — Pierce corpus exploration
This notebook measures the declared PDF directly. It writes no derived data and stores no generated output in Git.
The sample features follow the real development measurements: page word count, ink depth, orphan ink, and centre-line span.

In [ ]:
from pathlib import Path
import fitz
import cv2
import numpy as np

PDF = Path('data/raw/pierce-peoples-common-sense-medical-adviser-1890.pdf')
if not PDF.is_file():
    raise FileNotFoundError('Run scripts/get_data.sh before running this notebook')

def render(page, dpi=150):
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

doc = fitz.open(PDF)
word_counts = [len(page.get_text('words')) for page in doc]
print({'pages': len(doc), 'embedded_words': sum(word_counts), 'min_words': min(word_counts), 'max_words': max(word_counts)})


In [ ]:
sample_indices = sorted(set([0, 1, 11, 12, 73, 233, 512, 777, len(doc) - 1]))
sample_rows = []
for index in sample_indices:
    page = doc[index]
    gray = render(page)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ink = binary > 0
    paper = gray[~ink]
    background = float(np.median(paper)) if paper.size else 220.0
    ink_depth = background - float(np.percentile(gray[ink], 10)) if ink.any() else 0.0
    height, width = gray.shape
    covered = np.zeros((height, width), dtype=bool)
    sx, sy = width / page.rect.width, height / page.rect.height
    for x0, y0, x1, y1, *_ in page.get_text('words'):
        covered[int(y0 * sy):int(y1 * sy) + 1, int(x0 * sx):int(x1 * sx) + 1] = True
    orphan_ink = float((ink & ~covered).sum()) / max(int(ink.sum()), 1)
    centre = page.rect.width / 2
    words = page.get_text('words')
    span = sum(x0 < centre < x1 for x0, _, x1, *_ in words) / max(len(words), 1)
    sample_rows.append({'page_id': f'p{index + 1:04d}', 'words': len(words), 'ink_depth': round(ink_depth, 1), 'orphan_ink': round(orphan_ink, 4), 'centre_span': round(span, 4)})
sample_rows


Interpret the printed rows from this run, not hard-coded values. The current development measurements suggest predominantly single-column pages, uneven ink fade, and figure-adjacent orphan ink; those are hypotheses to verify on the run's actual sample.